# Statistical Hypothesis Testing: Customer Churn

This notebook validates the strongest EDA findings from `data/processed/model_dataset.csv` using simple, standard statistical hypothesis tests.

Scope:

- Statistical testing only.
- No modeling.
- No causal claims.
- All conclusions are interpreted as association in an observational churn dataset.

The target variable is `churn`, with values `Yes` and `No`. The default significance level is `alpha = 0.05`.

## Setup

Load the processed modeling dataset and import only the standard libraries needed for this hypothesis-testing step.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2_contingency, mannwhitneyu

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed" / "model_dataset.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.csv"
ALPHA = 0.05

## Load And Sanity Check Data

Confirm that the dataset is available, the required columns exist, and there are no missing values in the variables used for testing.

In [2]:
df = pd.read_csv(DATA_PATH)

required_columns = [
    "customer_id",
    "churn",
    "contract",
    "monthly_charges",
    "has_tech_support",
    "has_online_security",
]
missing_required_columns = [column for column in required_columns if column not in df.columns]
if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

analysis_columns = ["churn", "contract", "monthly_charges", "has_tech_support", "has_online_security"]
missing_values = df[analysis_columns].isna().sum()

print(f"Dataset path: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Duplicate customer_id rows: {df['customer_id'].duplicated().sum():,}")
print(f"Churn values: {sorted(df['churn'].unique())}")

display(pd.DataFrame({
    "column": analysis_columns,
    "dtype": df[analysis_columns].dtypes.astype(str).values,
    "missing_values": missing_values.values,
    "unique_values": df[analysis_columns].nunique(dropna=False).values,
}))

display(df[analysis_columns].head())

Dataset path: data\processed\model_dataset.csv
Rows: 7,032
Columns: 34
Duplicate customer_id rows: 0
Churn values: ['No', 'Yes']


,column,dtype,missing_values,unique_values
0,churn,object,0,2
1,contract,object,0,3
2,monthly_charges,float64,0,1584
3,has_tech_support,int64,0,2
4,has_online_security,int64,0,2


,churn,contract,monthly_charges,has_tech_support,has_online_security
0,No,Month-to-month,29.8500,0,0
1,No,One year,56.9500,0,1
2,Yes,Month-to-month,53.8500,0,1
3,No,One year,42.3000,1,1
4,Yes,Month-to-month,70.7000,0,0


## Statistical Approach

The tests below are bivariate checks against churn. They are useful for validating whether EDA patterns are unlikely to be random sampling noise, but they do not control for confounding variables.

For each test:

- `p < 0.05` means reject the null hypothesis.
- `p >= 0.05` means do not reject the null hypothesis.
- Rejecting the null supports an association, not a causal relationship.

In [3]:
test_summaries = []


def format_p_value(p_value):
    if p_value < 0.001:
        return f"{p_value:.3e}"
    return f"{p_value:.6f}"


def decision_from_p_value(p_value, alpha=ALPHA):
    return "Reject H0" if p_value < alpha else "Do not reject H0"


def cramers_v(contingency_table, chi2_statistic):
    n = contingency_table.to_numpy().sum()
    rows, columns = contingency_table.shape
    denominator = n * min(rows - 1, columns - 1)
    return np.sqrt(chi2_statistic / denominator) if denominator > 0 else np.nan


def labeled_index(table, labels=None):
    if labels is None:
        return table
    output = table.copy()
    output.index = output.index.map(lambda value: labels.get(value, value))
    return output


def run_chi_square(feature, business_name, takeaway, labels=None):
    counts = pd.crosstab(df[feature], df["churn"])
    churn_rates = pd.crosstab(df[feature], df["churn"], normalize="index").mul(100)
    chi2_statistic, p_value, dof, expected = chi2_contingency(counts)
    effect_size = cramers_v(counts, chi2_statistic)
    decision = decision_from_p_value(p_value)

    display(Markdown(f"### Observed Counts: {business_name}"))
    display(labeled_index(counts, labels))

    display(Markdown(f"### Row Percentages: {business_name}"))
    display(labeled_index(churn_rates.round(2), labels))

    result_table = pd.DataFrame({
        "test": ["Chi-square test of independence"],
        "chi2_statistic": [chi2_statistic],
        "degrees_of_freedom": [dof],
        "p_value": [p_value],
        "formatted_p_value": [format_p_value(p_value)],
        "cramers_v": [effect_size],
        "minimum_expected_count": [expected.min()],
        "decision_alpha_0_05": [decision],
    })
    display(result_table)

    display(Markdown(
        f"""
**Conclusion:** {decision} at alpha = {ALPHA:.2f}; the p-value is `{format_p_value(p_value)}`.  
**Business interpretation:** {takeaway}  
**Association warning:** This test supports an association between `{feature}` and churn, not a causal relationship.
""".strip()
    ))

    test_summaries.append({
        "business_question": business_name,
        "test": "Chi-square test of independence",
        "p_value": p_value,
        "formatted_p_value": format_p_value(p_value),
        "decision_alpha_0_05": decision,
        "effect_size": effect_size,
        "business_takeaway": takeaway,
    })


def run_mann_whitney():
    churned = df.loc[df["churn"].eq("Yes"), "monthly_charges"]
    retained = df.loc[df["churn"].eq("No"), "monthly_charges"]
    u_statistic, p_value = mannwhitneyu(churned, retained, alternative="two-sided", method="auto")
    rank_biserial = 2 * u_statistic / (len(churned) * len(retained)) - 1
    decision = decision_from_p_value(p_value)

    descriptive_stats = pd.DataFrame({
        "group": ["Churned customers", "Non-churned customers"],
        "n": [len(churned), len(retained)],
        "mean_monthly_charges": [churned.mean(), retained.mean()],
        "median_monthly_charges": [churned.median(), retained.median()],
        "q1": [churned.quantile(0.25), retained.quantile(0.25)],
        "q3": [churned.quantile(0.75), retained.quantile(0.75)],
    })
    display(descriptive_stats)

    result_table = pd.DataFrame({
        "test": ["Mann-Whitney U test"],
        "u_statistic": [u_statistic],
        "p_value": [p_value],
        "formatted_p_value": [format_p_value(p_value)],
        "rank_biserial_correlation": [rank_biserial],
        "decision_alpha_0_05": [decision],
    })
    display(result_table)

    takeaway = (
        "Churned customers have higher monthly charges in this dataset "
        f"(median ${churned.median():.2f}) than non-churned customers "
        f"(median ${retained.median():.2f}). This supports the EDA finding that charges differ by churn status."
    )

    display(Markdown(
        f"""
**Conclusion:** {decision} at alpha = {ALPHA:.2f}; the p-value is `{format_p_value(p_value)}`.  
**Business interpretation:** {takeaway}  
**Association warning:** This test supports an association between `monthly_charges` and churn, not a causal relationship.
""".strip()
    ))

    test_summaries.append({
        "business_question": "Do churned and non-churned customers differ significantly in monthly_charges?",
        "test": "Mann-Whitney U test",
        "p_value": p_value,
        "formatted_p_value": format_p_value(p_value),
        "decision_alpha_0_05": decision,
        "effect_size": rank_biserial,
        "business_takeaway": takeaway,
    })

## Test 1: Contract Type And Churn

**Business question:** Is churn significantly associated with contract type?

**Null hypothesis, H0:** Churn and contract type are independent.

**Alternative hypothesis, H1:** Churn and contract type are associated.

**Test choice and why:** Use a chi-square test of independence because both variables are categorical. The test checks whether the observed churn counts by contract type differ from what would be expected if churn and contract type were independent.

**Causality warning:** This test evaluates association only. It does not prove that contract type causes churn.

In [4]:
run_chi_square(
    feature="contract",
    business_name="Is churn significantly associated with contract type?",
    takeaway=(
        "Contract type is one of the strongest validated churn signals. "
        "Month-to-month customers show much higher churn than one-year and two-year customers, "
        "so contract structure should be reviewed as a core retention segmentation variable."
    ),
)

### Observed Counts: Is churn significantly associated with contract type?

churn,No,Yes
contract,,
Month-to-month,2220,1655
One year,1306,166
Two year,1637,48


### Row Percentages: Is churn significantly associated with contract type?

churn,No,Yes
contract,,
Month-to-month,57.2900,42.7100
One year,88.7200,11.2800
Two year,97.1500,2.8500


,test,chi2_statistic,degrees_of_freedom,p_value,formatted_p_value,cramers_v,minimum_expected_count,decision_alpha_0_05
0,Chi-square test of independence,1179.5458,2,0.0000,7.326e-257,0.4096,391.2355,Reject H0


**Conclusion:** Reject H0 at alpha = 0.05; the p-value is `7.326e-257`.  
**Business interpretation:** Contract type is one of the strongest validated churn signals. Month-to-month customers show much higher churn than one-year and two-year customers, so contract structure should be reviewed as a core retention segmentation variable.  
**Association warning:** This test supports an association between `contract` and churn, not a causal relationship.

## Test 2: Monthly Charges And Churn

**Business question:** Do churned and non-churned customers differ significantly in `monthly_charges`?

**Null hypothesis, H0:** The distribution of `monthly_charges` is the same for churned and non-churned customers.

**Alternative hypothesis, H1:** The distribution of `monthly_charges` differs between churned and non-churned customers.

**Test choice and why:** Use a two-sided Mann-Whitney U test because `monthly_charges` is numeric and churn creates two independent customer groups. This non-parametric test is appropriate for a skewed or non-normal charge distribution and does not require assuming normality.

**Causality warning:** This test evaluates association only. It does not prove that higher monthly charges cause churn.

In [5]:
run_mann_whitney()

,group,n,mean_monthly_charges,median_monthly_charges,q1,q3
0,Churned customers,1869,74.4413,79.6500,56.1500,94.2000
1,Non-churned customers,5163,61.3074,64.4500,25.1000,88.4750


,test,u_statistic,p_value,formatted_p_value,rank_biserial_correlation,decision_alpha_0_05
0,Mann-Whitney U test,5986148.5000,0.0000,8.467e-54,0.2407,Reject H0


**Conclusion:** Reject H0 at alpha = 0.05; the p-value is `8.467e-54`.  
**Business interpretation:** Churned customers have higher monthly charges in this dataset (median $79.65) than non-churned customers (median $64.45). This supports the EDA finding that charges differ by churn status.  
**Association warning:** This test supports an association between `monthly_charges` and churn, not a causal relationship.

## Test 3: Tech Support And Churn

**Business question:** Is churn significantly associated with `has_tech_support`?

**Null hypothesis, H0:** Churn and `has_tech_support` are independent.

**Alternative hypothesis, H1:** Churn and `has_tech_support` are associated.

**Test choice and why:** Use a chi-square test of independence because `has_tech_support` is a binary categorical feature and churn is categorical. The test checks whether churn counts differ more than expected between customers with and without the tech-support flag.

**Causality warning:** This test evaluates association only. It does not prove that tech support prevents churn.

In [6]:
run_chi_square(
    feature="has_tech_support",
    business_name="Is churn significantly associated with has_tech_support?",
    takeaway=(
        "Customers with the tech-support flag have a lower observed churn rate than customers without it. "
        "This makes tech support a useful service-engagement signal to carry into modeling and segmentation review."
    ),
    labels={0: "0 = does not have tech support flag", 1: "1 = has tech support flag"},
)

### Observed Counts: Is churn significantly associated with has_tech_support?

churn,No,Yes
has_tech_support,,
0 = does not have tech support flag,3433,1559
1 = has tech support flag,1730,310


### Row Percentages: Is churn significantly associated with has_tech_support?

churn,No,Yes
has_tech_support,,
0 = does not have tech support flag,68.7700,31.2300
1 = has tech support flag,84.8000,15.2000


,test,chi2_statistic,degrees_of_freedom,p_value,formatted_p_value,cramers_v,minimum_expected_count,decision_alpha_0_05
0,Chi-square test of independence,189.9668,1,0.0000,3.233e-43,0.1644,542.2014,Reject H0


**Conclusion:** Reject H0 at alpha = 0.05; the p-value is `3.233e-43`.  
**Business interpretation:** Customers with the tech-support flag have a lower observed churn rate than customers without it. This makes tech support a useful service-engagement signal to carry into modeling and segmentation review.  
**Association warning:** This test supports an association between `has_tech_support` and churn, not a causal relationship.

## Test 4: Online Security And Churn

**Business question:** Is churn significantly associated with `has_online_security`?

**Null hypothesis, H0:** Churn and `has_online_security` are independent.

**Alternative hypothesis, H1:** Churn and `has_online_security` are associated.

**Test choice and why:** Use a chi-square test of independence because `has_online_security` is a binary categorical feature and churn is categorical. The test checks whether churn counts differ more than expected between customers with and without the online-security flag.

**Causality warning:** This test evaluates association only. It does not prove that online security prevents churn.

In [7]:
run_chi_square(
    feature="has_online_security",
    business_name="Is churn significantly associated with has_online_security?",
    takeaway=(
        "Customers with the online-security flag have a lower observed churn rate than customers without it. "
        "This should be treated as a meaningful service-engagement or bundle signal before modeling."
    ),
    labels={0: "0 = does not have online security flag", 1: "1 = has online security flag"},
)

### Observed Counts: Is churn significantly associated with has_online_security?

churn,No,Yes
has_online_security,,
0 = does not have online security flag,3443,1574
1 = has online security flag,1720,295


### Row Percentages: Is churn significantly associated with has_online_security?

churn,No,Yes
has_online_security,,
0 = does not have online security flag,68.6300,31.3700
1 = has online security flag,85.3600,14.6400


,test,chi2_statistic,degrees_of_freedom,p_value,formatted_p_value,cramers_v,minimum_expected_count,decision_alpha_0_05
0,Chi-square test of independence,205.4158,1,0.0000,1.374e-46,0.1709,535.5567,Reject H0


**Conclusion:** Reject H0 at alpha = 0.05; the p-value is `1.374e-46`.  
**Business interpretation:** Customers with the online-security flag have a lower observed churn rate than customers without it. This should be treated as a meaningful service-engagement or bundle signal before modeling.  
**Association warning:** This test supports an association between `has_online_security` and churn, not a causal relationship.

## Consolidated Test Summary

The table below collects the p-values, decisions, effect-size context, and business takeaways from the four required tests.

In [8]:
summary_df = pd.DataFrame(test_summaries)
summary_df["effect_size"] = summary_df["effect_size"].round(4)
summary_df

,business_question,test,p_value,formatted_p_value,decision_alpha_0_05,effect_size,business_takeaway
0,Is churn significantly associated with contrac...,Chi-square test of independence,0.0000,7.326e-257,Reject H0,0.4096,Contract type is one of the strongest validate...
1,Do churned and non-churned customers differ si...,Mann-Whitney U test,0.0000,8.467e-54,Reject H0,0.2407,Churned customers have higher monthly charges ...
2,Is churn significantly associated with has_tec...,Chi-square test of independence,0.0000,3.233e-43,Reject H0,0.1644,Customers with the tech-support flag have a lo...
3,Is churn significantly associated with has_onl...,Chi-square test of independence,0.0000,1.374e-46,Reject H0,0.1709,Customers with the online-security flag have a...


## Limitations Before Modeling

- These tests are bivariate and do not control for confounders such as tenure, internet service type, payment method, bundle composition, or contract type.
- The dataset is observational, so every result should be interpreted as association, not causality.
- Large samples can produce very small p-values; business prioritization should consider effect size and segment size, not only statistical significance.
- The engineered service flags collapse several customer states into `0`; before modeling, compare them with the raw service columns and `internet_service` to avoid losing useful category detail.
- Multiple tests were run, which increases false-positive risk. The p-values here are extremely small, but a future broader testing program should define correction rules in advance.
- No modeling is performed in this notebook.